### Revision de cada batch de registros geolocalizados con el API de Google Maps


Aqui solo se unen los batches(62) de registros geolocalizados con el API de Google Maps

-  se les pega la info de marcia (mexico) por que me la salte en el paso 5, pero al final es necesaria para usar pdfs por que la confianza en el api es baja.

Y posteriormente se subdividen dada la confianza de cada geolocalizacion con el API.

In [1]:

# explorar valores

In [22]:
# # cargarlos  

# import pandas as pd
# from pathlib import Path

# # Ruta a la carpeta con los CSV
# ruta_batches = Path("5.batches")

# # Leer todos los archivos .csv de la carpeta
# archivos_csv = list(ruta_batches.glob("*.csv"))

# print(f"Se encontraron {len(archivos_csv)} archivos CSV")

# # Cargar y concatenar todos
# df_completo = pd.concat(
#     (pd.read_csv(archivo) for archivo in archivos_csv),
#     ignore_index=True,   # Reinicia el índice
#     axis=0
# )

# # clasificar nan
# df_completo.loc[df_completo["municipality"].isna(), "confidence_score"] = "Nan"

# df_completo.shape

In [24]:
# df_mexico = pd.read_csv("5.marcia_details_mexico.csv")

# df_mexico.shape

In [ ]:

# merged = pd.merge(
#     df_completo,
#     df_mexico[["id", "applicationNumber", "pdf_links"]],   # solo las columnas que necesitamos
#     on="id",                                                # clave común
#     how="left"                                              # left join (todas las filas de df_completo)
# )

# # Renombramos las columnas pegadas para que sea claro de dónde vienen (opcional pero recomendado)
# merged = merged.rename(columns={
#     "applicationNumber": "applicationNumber_marcia",
#     "pdf_links":         "pdf_links_marcia"
# })

# print(f"DataFrame original (df_completo): {len(df_completo):,} filas")
# print(f"DataFrame mergeado: {len(merged):,} filas\n")

# # ======================== CONTAR MATCHES Y UNMATCHES ========================
# matches = merged["applicationNumber_marcia"].notna().sum()
# unmatches = len(merged) - matches

# print(f"✅ MATCHES (id encontrado en df_mexico): {matches:,} filas ({matches/len(merged)*100:.2f}%)")
# print(f"❌ UNMATCHES (id NO encontrado):         {unmatches:,} filas ({unmatches/len(merged)*100:.2f}%)")

# # ======================== VERIFICACIÓN RÁPIDA ========================
# print("\nPrimeras filas del merge (para validar):")
# display(merged[["id", "municipality", "confidence_score", 
#                 "applicationNumber_marcia", "pdf_links_marcia"]].head(8))

# # Opcional: ver cuántos ids únicos hay en cada dataframe (para diagnosticar)
# print(f"\nIDs únicos en df_completo: {df_completo['id'].nunique():,}")
# print(f"IDs únicos en df_mexico:    {df_mexico['id'].nunique():,}")

DataFrame original (df_completo): 61,920 filas
DataFrame mergeado: 61,920 filas

✅ MATCHES (id encontrado en df_mexico): 61,916 filas (99.99%)
❌ UNMATCHES (id NO encontrado):         4 filas (0.01%)

Primeras filas del merge (para validar):


,id,municipality,confidence_score,applicationNumber_marcia,pdf_links_marcia
0,RM202202853310,Ciudad de México,1.0,2853310,https://acervomarcas.impi.gob.mx:8181/marcanet...
1,RM202202853311,Prolongación Lázaro Cárdenas,0.4,2853311,https://acervomarcas.impi.gob.mx:8181/marcanet...
2,RM202202853312,San José de los Olvera,1.0,2853312,https://acervomarcas.impi.gob.mx:8181/marcanet...
3,RM202202853321,Guadalajara,1.0,2853321,https://acervomarcas.impi.gob.mx:8181/marcanet...
4,RM202202853330,Santiago de Querétaro,1.0,2853330,https://acervomarcas.impi.gob.mx:8181/marcanet...
5,RM202202853334,Naucalpan de Juárez,1.0,2853334,https://acervomarcas.impi.gob.mx:8181/marcanet...
6,RM202202853350,Coatzacoalcos,1.0,2853350,https://acervomarcas.impi.gob.mx:8181/marcanet...
7,RM202202853351,Mérida,0.65,2853351,https://acervomarcas.impi.gob.mx:8181/marcanet...



IDs únicos en df_completo: 61,920
IDs únicos en df_mexico:    103,094


In [61]:
print(merged["id"].nunique())

print("---------------------------------------------------------")

print(merged["confidence_score"].value_counts())
print(f"Valores NA en 'confidence_score': {merged['confidence_score'].isna().sum()}")

print("---------------------------------------------------------")

#nan counts
print(merged["municipality"].isna().sum())

61920
---------------------------------------------------------
confidence_score
1.0     37218
0.4      8456
0.9      6928
0.65     3784
Nan      3612
0.42      801
0.6       575
0.7       546
Name: count, dtype: int64
Valores NA en 'confidence_score': 0
---------------------------------------------------------
3612


In [ ]:
#merged.to_csv("6.geolocalizados_completo_1_ronda.csv", index=False, encoding="utf-8-sig")

## dividir por confianza 

aca abria que cargar el objeto df_completo a partir de #df_completo.to_csv("6.geolocalizados_completo_1_ronda.csv", index=False, encoding="utf-8-sig")

para actualizar cuando no termine el scraper de acervo

este no importa tanto, el siguiente paso usa 6.geolocalizados_completo_1_ronda.csv

In [44]:
# import pandas as pd
# from pathlib import Path

# # ==================== CONFIGURACIÓN ====================
# output_folder = Path("6.data_integretation_checkpoint_data")
# output_folder.mkdir(parents=True, exist_ok=True)

# print("Iniciando guardado de subdivisiones por confidence_score...\n")

# for confidence_value, subdf in df_completo.groupby("confidence_score", dropna=False):
    
#     # ==================== Nombre de archivos ====================
#     if pd.isna(confidence_value):
#         base_name = "confidence_score_NaN"
#     else:
#         # Ej: 1.0 → confidence_score_1_0
#         base_name = f"confidence_score_{str(confidence_value).replace('.', '_')}"
    
#     csv_filename = f"{base_name}.csv"
#     txt_filename = f"{base_name}_ids.txt"
    
#     csv_path = output_folder / csv_filename
#     txt_path = output_folder / txt_filename
    
#     # ==================== Guardar CSV ====================
#     subdf.to_csv(csv_path, index=False)
    
#     # ==================== Guardar lista de IDs en TXT ====================
#     if "id" in subdf.columns:
#         with open(txt_path, "w", encoding="utf-8") as f:
#             for idx in subdf["id"]:
#                 f.write(str(idx) + "\n")
        
#         print(f"✓ Guardado: {csv_filename:<35} → {len(subdf):,} registros | "
#               f"{txt_filename} ({subdf['id'].nunique():,} ids únicos)")
#     else:
#         print(f"✓ Guardado: {csv_filename:<35} → {len(subdf):,} registros | "
#               f"(No se encontró columna 'id' para TXT)")

# print("\n¡Proceso terminado!")
# print(f"Todos los archivos se guardaron en: {output_folder.resolve()}")

### Aqui esta la base de todo el documento de 103mil registros correspondientes a mexico (61mil + ubicaciones diferente) 

In [69]:
import pandas as pd

df_mexico = pd.read_csv("5.marcia_details_mexico.csv")

df_ids_mx = pd.read_csv("5.mexico_addresses_id.csv")


merged = pd.merge(
    df_mexico,
    df_ids_mx[["id", "Addr", "Addr_id"]],   # solo las columnas que necesitamos
    on="id",                         # clave común
    how="left"                       # left join (todas las filas de df_mexico)
)

merged = merged[['id', 'title', 'applicationNumber', 'registrationNumber',
       'applicationDate', 'publicationDt', 'registrationDate', 'expiryDate',
       'appType', 'classes', 'Addr_x', 'Cry', 'Name', 'email', 'pdf_links', 'Addr_id']]
#, 'startDate', 'receptionYear','dateOfConclusion', 'viennaCodes', 'Addr_y'

merged.sort_values(by="Addr_id", inplace=True)

#merged.to_csv("6.marcia_mexico_pre_data_merge_og.csv", index=False, encoding="utf-8-sig")


/tmp/ipykernel_74219/4071497544.py:3: DtypeWarning: Columns (3,4,10,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df_mexico = pd.read_csv("5.marcia_details_mexico.csv")


In [64]:
print(merged.columns)

Index(['id', 'title', 'applicationNumber', 'registrationNumber',
       'applicationDate', 'publicationDt', 'registrationDate', 'expiryDate',
       'appType', 'classes', 'Addr_x', 'Cry', 'Name', 'email', 'pdf_links',
       'Addr_y', 'Addr_id'],
      dtype='object')


In [67]:
merged.tail(10)

,id,title,applicationNumber,registrationNumber,applicationDate,publicationDt,registrationDate,expiryDate,appType,classes,Addr_x,Cry,Name,email,pdf_links,Addr_id
103085,RM202303030567,ABEJAS AL VUELO,3030567,2632403,2/10/2023,06/10/2023,29/11/2023,29/11/2033,REGISTRO DE MARCA,30,"CALKINI NUM. EXT. 163, LOMAS DE PADIERNA",MEXICO,PAULINA FABIOLA SALAS LOPEZ,NaN,https://acervomarcas.impi.gob.mx:8181/marcanet...,61915.0
103086,RM202303031150,MANOS DE QUILTEPEC,3031150,2632404,3/10/2023,10/10/2023,29/11/2023,29/11/2033,REGISTRO DE MARCA,29,"QUILTEPEC EL MILAGRO NUM. EXT. 11, SAN ANDRES ...",MEXICO,ROSA SORIANO,NaN,https://acervomarcas.impi.gob.mx:8181/marcanet...,61916.0
103089,RM202303039045,ALEGRILLÍN,3039045,2632405,17/10/2023,24/10/2023,29/11/2023,29/11/2033,REGISTRO DE MARCA,30,"VICENTE GUERRERO NUM. EXT. 37, CALYECAC TULYEH...",MEXICO,JOSE MOLOTLA ROJAS,NaN,https://acervomarcas.impi.gob.mx:8181/marcanet...,61917.0
103093,RM202303044525,GOLDAMARANTO,3044525,2639685,26/10/2023,03/11/2023,12/12/2023,12/12/2033,REGISTRO DE MARCA,30,"VICENTE GUERRERO NUM. EXT. 37, CALYECAC TULYE...",MEXICO,JAVIER MOLOTLA ROJAS,NaN,https://acervomarcas.impi.gob.mx:8181/marcanet...,61917.0
103090,RM202303039379,AQUABERRIES,3039379,2632406,17/10/2023,24/10/2023,29/11/2023,29/11/2033,REGISTRO DE MARCA,31,"FLORICULTOR NUM. EXT. 43, SAN LUIS TLAXIALTEMALCO",MEXICO,NOEL OLAF NIETO FLORES,NaN,https://acervomarcas.impi.gob.mx:8181/marcanet...,61918.0
103091,RM202303042458,GRANJA LA ABEJITA CONTENTA,3042458,2639684,23/10/2023,30/10/2023,12/12/2023,12/12/2033,REGISTRO DE MARCA,33,PROLONGACION JOSEFA ORTIZ DE DOMINGUEZ NUM. EX...,MEXICO,FEDERICO PALMA BALDERRAMA,NaN,https://acervomarcas.impi.gob.mx:8181/marcanet...,61919.0
7400,2711240,2502972,14/3/2022,22/03/2022,01/02/2023,01/02/2033,REGISTRO DE MARCA,5,"27.01.01, 27.01.12, 19.13.01, 19.13.22, 02.09....",RETORNO DE LOS CARRETONES NUM. EXT. 33 NUM. IN...,WELLNESS PHARMA S DE R.L DE C.V,NaN,16/02/2023,2022,NaN,NaN
20152,2739259,2520443,6/5/2022,13/05/2022,14/03/2023,14/03/2033,REGISTRO DE MARCA,35,"27.05.01, 27.05.09, 27.05.10, 16.01.01, 07.01.06","GEMA NUM. EXT. S/N, PIEDRAS LISAS",OMAR EDIBALDO VAZQUEZ DURAN,NaN,27/03/2023,2022,NaN,NaN
39008,2763456,2549097,7/6/2022,14/06/2022,22/05/2023,22/05/2033,REGISTRO DE MARCA,30,"27.05.01, 27.05.07, 27.05.17, 26.01.01, 26.01....","LOS ALDAMA NUM. EXT. 603, CENTRO",TANIA GUTIERREZ JIMENEZ,NaN,15/06/2023,2022,NaN,NaN
92031,2845139,2584822,9/11/2022,16/11/2022,10/08/2023,10/08/2033,REGISTRO DE MARCA,44,"27.05.01, 27.05.05, 27.05.10, 27.05.17, 27.05....","CORDOBA NUM. EXT. 183 NUM. INT. 103, ROMA NORTE",JORGE LUIS DE LEON RENDON,NaN,17/10/2023,2022,NaN,NaN
